In [ ]:
from keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

Task 1.1 - Load MNIST dataset (10,000 training samples; 2,000 test samples)

In [ ]:
X_train = X_train[:10000]
y_train = y_train[:10000]

X_test = X_test[:2000]
y_test = y_test[:2000]

Task 1.2 - Display at least 5 images with labels

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.show()

Task 1.3 - Explain: What the dataset represents? Why image data is different from tabular data?

MNIST is a collection of handwritten digit images. The image data is different from tabular data because structure matters. Pixels are spatially related in images. Shape is also different (3D as opposed to 2D). Raw values are also not meaningful alone. Model performance also differs with tabular data.

Task 1.4 - Flatten images (28 x 28 to 784)

In [ ]:
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

Task 1.5 - Normalize pixel values (by dividing pixel values with 255)

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

Task 1.6 - Explain why normalization is required.

Normaliztion is needed because of faster convergence, equal weight, avoiding numerical issues, and consistency. Gradient descent works better when input features are on a small, consistent scale. Without normalization, the model may treat high pixel values as more important. Also, L models assume inputs are centered around 0 or within a small range. Diving by 255 puts everything on the same scale.

Task 1.7 - Model Training:
• KNN
• SVM
• Decision Tree
• Random Forest

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# SVM
svm = SVC(kernel='rbf')
svm.fit(X_train, y_train)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

models = {'KNN': knn, 'SVM': svm, 'Decision Tree': dt, 'Random Forest': rf}

for name, model in models.items():
    score = model.score(X_test, y_test)
    print(f'{name} Accuracy: {score:.4f}')

Task 1.8 - Compute for EACH model:
• Accuracy
• Confusion Matrix
• Classification Report

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

for name, model in models.items():
    y_pred = model.predict(X_test)

    print(f"{'='*40}")
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

Task 1.9 - Interpret Results:
• Which model performed best
• Which struggled and why

Best Performer: SVM (94.45%)
SVM achieved the highest accuracy and the most consistent precision/recall across all digits (mostly 0.92–0.99). Its confusion matrix is the cleanest as the errors are few and scattered. SVM excels here because it finds an optimal decision boundary in high-dimensional space (784 features), which suits flattened image data well.

Runner-up: Random Forest (93.30%)
Strong performance with balanced precision/recall. Slightly behind SVM because random forests make decisions based on individual pixel values independently, missing some spatial relationships between pixels.

Solid: KNN (92.35%)
Performs well but is slower to predict because it must compare each test image against all 10,000 training images. Notice digit 8 had the lowest recall (0.80), meaning it frequently misclassified 8s as other digits. 8s visually overlap with 3, 5, and 9, which confuses distance-based methods.

Worst: Decision Tree (76.70%)
Significantly underperformed at only 76.70%. Key observations:

Digit 8 was the worst (F1: 0.61) — heavily confused with 3, 5, and 6
Digit 5 also struggled (F1: 0.68)
The confusion matrix shows errors scattered everywhere, not concentrated

Why Decision Tree struggled: A single tree makes rigid, axis-aligned splits on individual pixel values. It cannot capture the complex, curved patterns in handwritten digits and tends to overfit to specific training examples rather than learning generalizable shape features. Random Forest fixes this by averaging many trees, which is why it scores ~17% higher.

Summary ranking: SVM > Random Forest > KNN > Decision Tree

Task 1.10 - Perform GridSearchCV for ALL models
Use following parameter grids:
• KNN: n_neighbors = [3,5,7, 10]
• SVM: C = [0.1,1,10], kernel = ['linear','rbf']
• Decision Tree: max_depth = [None,5,10]
• Random Forest: n_estimators = [50,100, 200, 300], max_depth = [None,5,10]

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 10]}
    },
    'SVM': {
        'model': SVC(),
        'params': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(),
        'params': {'max_depth': [None, 5, 10]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(),
        'params': {'n_estimators': [50, 100, 200, 300], 'max_depth': [None, 5, 10]}
    }
}

best_models = {}

for name, config in param_grids.items():
    print(f"Running GridSearchCV for {name}...")
    grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    print(f"Best Params: {grid.best_params_}")
    print(F"Best CV Accuracy: {grid.best_score_:.4f}\n")

Task 1.11 - Report for each model:
• Best parameters
• Best cross-validation score

Model Comparison Report – MNIST Digit Classification

KNN (K-Nearest Neighbors)
- **Best Parameters:** `n_neighbors = 3`
- **Best Cross-Validation Accuracy:** 93.95%
- **Notes:** The smallest neighbor count (3) performed best, meaning tighter local boundaries
  capture digit similarities more accurately. Larger values like 10 diluted predictions by
  including less similar neighbors.

---

SVM (Support Vector Machine)
- **Best Parameters:** `C = 10, kernel = rbf`
- **Best Cross-Validation Accuracy:** 96.62%
- **Notes:** The RBF kernel outperformed linear, indicating digit boundaries are non-linear.
  A high C value (10) means the model prioritizes minimizing misclassifications over maximizing
  margin width, which paid off on this dataset.

---

Decision Tree
- **Best Parameters:** `max_depth = 10`
- **Best Cross-Validation Accuracy:** 80.42%
- **Notes:** Limiting depth to 10 reduced overfitting compared to an unconstrained tree.
  Still the weakest model. A single tree cannot capture the complexity of handwritten digits
  regardless of tuning.

---

Random Forest
- **Best Parameters:** `n_estimators = 300, max_depth = None`
- **Best Cross-Validation Accuracy:** 94.97%
- **Notes:** More trees (300) improved stability through better ensemble averaging.
  Unrestricted depth (None) allows each tree to fully learn training patterns, with
  overfitting controlled by the ensemble rather than depth pruning.

---

Overall Ranking

| Rank | Model         | Best CV Accuracy |
|------|---------------|-----------------|
| 1    | SVM           | 96.62%          |
| 2    | Random Forest | 94.97%          |
| 3    | KNN           | 93.95%          |
| 4    | Decision Tree | 80.42%          |

**Best Model: SVM** with `C=10` and `kernel=rbf` achieved the highest accuracy
and the most consistent per-digit performance across all classes.


Task 1.12 - Evaluate tuned models on test set

In [ ]:
print("Tuned Model Evaluation on Test Set")
print("="*40)

for name, model in best_models.items():
    y_pred = model.predict(X_test)

    print(f"\nModel: {name}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
    plt.title(f'{name} (Tuned) - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

Task 1.13 - Identify best overall model and justify your choice using metrics

Best Overall Model: SVM (Tuned)

Test Set Performance Summary

| Model         | Test Accuracy | Macro F1 | Macro Precision | Macro Recall |
|---------------|---------------|----------|-----------------|--------------|
| KNN           | 92.35%        | 0.92     | 0.92            | 0.92         |
| **SVM**       | **95.55%**    | **0.96** | **0.96**        | **0.96**     |
| Decision Tree | 77.40%        | 0.77     | 0.77            | 0.77         |
| Random Forest | 93.10%        | 0.93     | 0.93            | 0.93         |

---

Justification

1. Highest Accuracy
SVM scored **95.55%** on the test set, outperforming all other models by at least
2.45 percentage points.

2. Most Consistent Per-Class Performance
Every digit scored between **0.93–0.98 F1**. No digit fell below 0.93, meaning SVM
classifies all digits reliably — not just the easy ones.

3. Cleanest Confusion Matrix
Off-diagonal values are minimal and evenly scattered, with no single digit pair
causing systematic confusion.

4. Best Generalization
CV accuracy (96.62%) and test accuracy (95.55%) are close together, indicating the
model generalizes well to unseen data without overfitting.

5. Tuning Had the Most Impact
Accuracy improved from 94.45% → 95.55% (+1.1%) after GridSearch, confirming that
`C=10, kernel=rbf` was meaningfully better than default parameters.

---

Why Not Random Forest?
Random Forest is competitive but trails SVM by **2.45%** and showed a slight drop
after tuning (93.30% → 93.10%), suggesting it was already near its ceiling with
this dataset size.

---

Conclusion
The tuned SVM with `C=10` and `kernel=rbf` is the best overall model for MNIST digit
classification, offering the highest accuracy, most balanced per-class metrics, and
the strongest generalization to unseen data.


Task 1.14 - Display at least 5 test images. Show - Predicted label, Actual label

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

best_svm = best_models['SVM']
y_pred = best_svm.predict(X_test)

for i, ax in enumerate(axes):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Actual: {y_test[i]}\nPredicted: {y_pred[i]}', 
                 color='green' if y_pred[i] == y_test[i] else 'red')
    ax.axis('off')

plt.suptitle('SVM - Sample Test Predictions', fontsize=14)
plt.tight_layout()
plt.show()